# Notebook 04: 学習と推論（テキスト生成）

これまで構築したモデルを「学習」させ「テキストを生成」します。

**このノートでカバーする内容**：

1. **損失関数**：Cross-Entropy Loss の実際の計算
2. **勾配の直感**：数値微分で「勾配とは何か」を体感
3. **パラメータ更新**：SGD（確率的勾配降下法）の 1 ステップ
4. **自己回帰生成**：トークンを 1 つずつ生成する仕組み
5. **サンプリング戦略**：Greedy / Temperature / Top-k の比較

**使うライブラリ**：numpy のみ

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# ======================================================
# Notebook 01-03 の全コンポーネントを再定義
# ======================================================

vocab_size = 8
d_model    = 4
n_layers   = 2
num_heads  = 2
d_ff       = 8

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

def gelu(x):
    return 0.5 * x * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)))

def positional_encoding(seq_len, d_model):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            angle = pos / (10000 ** (2 * i / d_model))
            PE[pos, i]   = np.sin(angle)
            if i + 1 < d_model:
                PE[pos, i+1] = np.cos(angle)
    return PE

def scaled_dot_product_attention(Q, K, V, mask=True):
    seq_len, d_k = Q.shape
    scores = Q @ K.T / np.sqrt(d_k)
    if mask:
        causal = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
        scores[causal] = -1e9
    return softmax(scores, axis=-1) @ V


class MultiHeadAttention:
    def __init__(self, d_model, num_heads, seed=0):
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        rng = np.random.default_rng(seed)
        self.W_Qs = [rng.standard_normal((d_model, self.d_k)) * 0.5 for _ in range(num_heads)]
        self.W_Ks = [rng.standard_normal((d_model, self.d_k)) * 0.5 for _ in range(num_heads)]
        self.W_Vs = [rng.standard_normal((d_model, self.d_k)) * 0.5 for _ in range(num_heads)]
        self.W_O  = rng.standard_normal((d_model, d_model)) * 0.5

    def forward(self, X):
        heads = []
        for h in range(self.num_heads):
            Q_h = X @ self.W_Qs[h]
            K_h = X @ self.W_Ks[h]
            V_h = X @ self.W_Vs[h]
            heads.append(scaled_dot_product_attention(Q_h, K_h, V_h))
        return np.concatenate(heads, axis=-1) @ self.W_O


class LayerNorm:
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)
        self.beta  = np.zeros(d_model)
        self.eps   = eps

    def forward(self, x):
        mean = x.mean(axis=-1, keepdims=True)
        var  = x.var(axis=-1, keepdims=True)
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta


class FeedForward:
    def __init__(self, d_model, d_ff, seed=0):
        rng = np.random.default_rng(seed)
        self.W1 = rng.standard_normal((d_model, d_ff)) * 0.3
        self.b1 = np.zeros(d_ff)
        self.W2 = rng.standard_normal((d_ff, d_model)) * 0.3
        self.b2 = np.zeros(d_model)

    def forward(self, x):
        return gelu(x @ self.W1 + self.b1) @ self.W2 + self.b2


class TransformerBlock:
    def __init__(self, d_model, num_heads, d_ff, seed=0):
        self.mha = MultiHeadAttention(d_model, num_heads, seed=seed)
        self.ffn = FeedForward(d_model, d_ff, seed=seed+1)
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)

    def forward(self, X):
        X = self.ln1.forward(X + self.mha.forward(X))
        X = self.ln2.forward(X + self.ffn.forward(X))
        return X


# モデルの初期化
np.random.seed(42)
E    = np.random.randn(vocab_size, d_model)      # 埋め込みテーブル
blocks = [TransformerBlock(d_model, num_heads, d_ff, seed=layer*10) for layer in range(n_layers)]
np.random.seed(99)
W_lm = np.random.randn(d_model, vocab_size) * 0.3  # LM ヘッド


def model_forward(token_ids_list):
    """フォワードパス全体: token_ids → logits"""
    ids = np.array(token_ids_list)
    seq = len(ids)
    X = E[ids] + positional_encoding(seq, d_model)
    for block in blocks:
        X = block.forward(X)
    return X @ W_lm, X   # logits と最終隠れ状態を返す


# テスト
tokens = [2, 5, 1, 3]
logits, hidden = model_forward(tokens)
print(f"モデル初期化完了")
print(f"  入力 token_ids: {tokens}")
print(f"  logits shape:   {logits.shape}")
print(f"  hidden shape:   {hidden.shape}")

モデル初期化完了
  入力 token_ids: [2, 5, 1, 3]
  logits shape:   (4, 8)
  hidden shape:   (4, 4)


---
## 1. 言語モデルの学習タスク

**次トークン予測**：入力系列の各位置で「次のトークンを予測する」タスクです。

```
入力:   [A, B, C, D]
正解:   [B, C, D, E]    ← 各位置の「次のトークン」
```

この形式では 1 つの系列から `seq_len` 個の学習サンプルが得られます（効率的！）

In [2]:
# 入力と正解ラベルの準備
# 入力: [2, 5, 1, 3] → 正解: [5, 1, 3, 6]（各位置の「次のトークン」）
input_ids  = [2, 5, 1, 3]
target_ids = [5, 1, 3, 6]   # 最後は仮の次トークン

print("学習の対応関係:")
for i, (inp, tgt) in enumerate(zip(input_ids, target_ids)):
    print(f"  位置 {i}: 入力 token_id={inp} → 正解 token_id={tgt}")

# フォワードパス
logits, hidden = model_forward(input_ids)
probs = softmax(logits, axis=-1)

print()
print(f"logits: shape = {logits.shape}")
print(logits)
print()
print(f"probs （softmax 後）: shape = {probs.shape}")
print(probs)

学習の対応関係:
  位置 0: 入力 token_id=2 → 正解 token_id=5
  位置 1: 入力 token_id=5 → 正解 token_id=1
  位置 2: 入力 token_id=1 → 正解 token_id=3
  位置 3: 入力 token_id=3 → 正解 token_id=6

logits: shape = (4, 8)
[[ 0.3229 -1.2574  0.0663  0.1073 -0.165  -0.1017 -0.0456 -0.7564]
 [ 0.2469  0.6681  0.418   0.5493 -0.2005 -0.4205  0.4235  0.1304]
 [-0.0241  1.7764  0.0752  0.1367 -0.0624 -0.1303  0.0518  0.9722]
 [-0.2819  1.6696  0.1205  0.126   0.1177 -0.0671  0.2709  0.8788]]

probs （softmax 後）: shape = (4, 8)
[[0.1966 0.0405 0.1521 0.1585 0.1207 0.1286 0.136  0.0668]
 [0.1205 0.1836 0.143  0.163  0.077  0.0618 0.1438 0.1072]
 [0.0668 0.404  0.0737 0.0784 0.0642 0.06   0.072  0.1808]
 [0.0535 0.3764 0.08   0.0804 0.0797 0.0663 0.0929 0.1707]]


---
## 2. Cross-Entropy Loss（交差エントロピー損失）

正解トークンを正しく予測できているかを測ります：

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i \mid x_{<i})$$

- $P(y_i \mid x_{<i})$：正解トークンに対してモデルが割り当てた確率
- 正解確率が高いほど $\log P$ は 0 に近く、損失は小さい
- 正解確率が低いほど $\log P$ は大きな負の値になり、損失は大きい

In [3]:
# ステップごとに損失を計算
print("=" * 55)
print("Cross-Entropy Loss の計算")
print("=" * 55)

loss_per_pos = []
for i, target in enumerate(target_ids):
    p_correct = probs[i, target]          # 正解トークンの確率
    loss_i    = -np.log(p_correct + 1e-9) # -log(P)
    loss_per_pos.append(loss_i)
    print(f"位置 {i}: 正解 ID={target}")
    print(f"  正解の確率 P(y={target}) = {p_correct:.6f}")
    print(f"  -log(P)               = {loss_i:.6f}")
    print()

total_loss = np.mean(loss_per_pos)
print(f"平均 Cross-Entropy Loss = {total_loss:.6f}")

Cross-Entropy Loss の計算
位置 0: 正解 ID=5
  正解の確率 P(y=5) = 0.128614
  -log(P)               = 2.050939

位置 1: 正解 ID=1
  正解の確率 P(y=1) = 0.183606
  -log(P)               = 1.694965

位置 2: 正解 ID=3
  正解の確率 P(y=3) = 0.078401
  -log(P)               = 2.545915

位置 3: 正解 ID=6
  正解の確率 P(y=6) = 0.092949
  -log(P)               = 2.375705

平均 Cross-Entropy Loss = 2.166881


In [4]:
# ベクトル化版（まとめて計算）
def cross_entropy_loss(logits, targets):
    probs    = softmax(logits, axis=-1)
    n        = len(targets)
    log_p    = np.log(probs[np.arange(n), targets] + 1e-9)
    return -log_p.mean(), probs

loss, probs_check = cross_entropy_loss(logits, target_ids)
print(f"ベクトル化計算（確認）: loss = {loss:.6f}")
print()

# 損失の解釈
uniform_loss = np.log(vocab_size)   # ランダム予測時の損失
print(f"ランダム予測の場合の損失 = -log(1/{vocab_size}) = log({vocab_size}) = {uniform_loss:.4f}")
print(f"現在の損失 = {loss:.4f}")
print(f"→ ランダムより {'良い' if loss < uniform_loss else '悪い'} 予測")

ベクトル化計算（確認）: loss = 2.166881

ランダム予測の場合の損失 = -log(1/8) = log(8) = 2.0794
現在の損失 = 2.1669
→ ランダムより 悪い 予測


---
## 3. 勾配の直感：数値微分

**勾配**とは「重みを少し動かしたとき、損失がどれくらい変わるか」です：

$$\frac{\partial \mathcal{L}}{\partial w} \approx \frac{\mathcal{L}(w + \varepsilon) - \mathcal{L}(w - \varepsilon)}{2\varepsilon}$$

- 勾配が正 → w を大きくすると損失が増える → w を小さくするべき
- 勾配が負 → w を大きくすると損失が減る → w を大きくするべき

In [5]:
# W_lm の特定の重みに対する数値微分
eps = 1e-4

def compute_loss(W_lm_candidate):
    """W_lm を変えたときの損失を計算"""
    log_i, hid = model_forward(input_ids)
    log_candidate = hid @ W_lm_candidate
    l, _ = cross_entropy_loss(log_candidate, target_ids)
    return l

print("数値微分: W_lm[0, 0] に対する勾配")
print()

# W_lm[0, 0] を +ε / -ε 変化させて損失の差を測る
W_plus  = W_lm.copy(); W_plus[0, 0]  += eps
W_minus = W_lm.copy(); W_minus[0, 0] -= eps

loss_plus  = compute_loss(W_plus)
loss_minus = compute_loss(W_minus)
grad_numerical = (loss_plus - loss_minus) / (2 * eps)

print(f"W_lm[0,0] = {W_lm[0,0]:.6f}")
print(f"W_lm[0,0] + {eps}: 損失 = {loss_plus:.6f}")
print(f"W_lm[0,0] - {eps}: 損失 = {loss_minus:.6f}")
print(f"数値微分による勾配 = ({loss_plus:.6f} - {loss_minus:.6f}) / (2×{eps})")
print(f"               = {grad_numerical:.6f}")

数値微分: W_lm[0, 0] に対する勾配

W_lm[0,0] = -0.042708
W_lm[0,0] + 0.0001: 損失 = 2.166884
W_lm[0,0] - 0.0001: 損失 = 2.166878
数値微分による勾配 = (2.166884 - 2.166878) / (2×0.0001)
               = 0.032168


---
## 4. 解析的な勾配（Softmax + Cross-Entropy）

Softmax と Cross-Entropy を組み合わせると、logits に対する勾配は非常にシンプルです：

$$\frac{\partial \mathcal{L}}{\partial \text{logits}_{i,j}} = \frac{1}{N}(P_{i,j} - \mathbf{1}[j = y_i])$$

つまり **「予測確率 − 正解ラベル」** がそのまま勾配になります。

In [6]:
# ∂L/∂logits を計算
n = len(target_ids)
one_hot = np.zeros_like(probs)
one_hot[np.arange(n), target_ids] = 1.0   # 正解位置に 1

grad_logits = (probs - one_hot) / n        # (seq_len, vocab_size)

print("probs（予測確率）:")
print(probs)
print()
print("one_hot（正解ラベル）:")
print(one_hot)
print()
print("∂L/∂logits = (probs - one_hot) / N:")
print(grad_logits)
print()
print("解釈:")
for i, tgt in enumerate(target_ids):
    print(f"  位置 {i}: 正解 ID={tgt} の勾配={grad_logits[i, tgt]:.6f}")
    print(f"    → 正解確率を上げるために logits[{i},{tgt}] を増やす方向")

probs（予測確率）:
[[0.1966 0.0405 0.1521 0.1585 0.1207 0.1286 0.136  0.0668]
 [0.1205 0.1836 0.143  0.163  0.077  0.0618 0.1438 0.1072]
 [0.0668 0.404  0.0737 0.0784 0.0642 0.06   0.072  0.1808]
 [0.0535 0.3764 0.08   0.0804 0.0797 0.0663 0.0929 0.1707]]

one_hot（正解ラベル）:
[[0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0.]]

∂L/∂logits = (probs - one_hot) / N:
[[ 0.0492  0.0101  0.038   0.0396  0.0302 -0.2178  0.034   0.0167]
 [ 0.0301 -0.2041  0.0357  0.0408  0.0193  0.0155  0.0359  0.0268]
 [ 0.0167  0.101   0.0184 -0.2304  0.0161  0.015   0.018   0.0452]
 [ 0.0134  0.0941  0.02    0.0201  0.0199  0.0166 -0.2268  0.0427]]

解釈:
  位置 0: 正解 ID=5 の勾配=-0.217846
    → 正解確率を上げるために logits[0,5] を増やす方向
  位置 1: 正解 ID=1 の勾配=-0.204099
    → 正解確率を上げるために logits[1,1] を増やす方向
  位置 2: 正解 ID=3 の勾配=-0.230400
    → 正解確率を上げるために logits[2,3] を増やす方向
  位置 3: 正解 ID=6 の勾配=-0.226763
    → 正解確率を上げるために logits[3,6] を増やす方向


In [7]:
# ∂L/∂W_lm = hidden^T @ ∂L/∂logits
grad_W_lm = hidden.T @ grad_logits   # (d_model, vocab_size)

print(f"∂L/∂W_lm: shape = {grad_W_lm.shape}")
print(grad_W_lm)
print()

# 数値微分との比較確認
grad_analytical_00 = grad_W_lm[0, 0]
print(f"解析的勾配 ∂L/∂W_lm[0,0] = {grad_analytical_00:.6f}")
print(f"数値微分  ∂L/∂W_lm[0,0] = {grad_numerical:.6f}")
print(f"差 = {abs(grad_analytical_00 - grad_numerical):.2e}  ← ε^2 オーダー")

∂L/∂W_lm: shape = (4, 8)
[[ 0.0322  0.017   0.0565 -0.2257  0.0422  0.2114 -0.256   0.1223]
 [ 0.0221 -0.2965 -0.0009  0.3971 -0.0095 -0.3024  0.2877 -0.0976]
 [ 0.0403 -0.1436  0.0297  0.0196  0.0167 -0.1661  0.2086 -0.0052]
 [-0.0945  0.4231 -0.0852 -0.191  -0.0495  0.257  -0.2404 -0.0195]]

解析的勾配 ∂L/∂W_lm[0,0] = 0.032168
数値微分  ∂L/∂W_lm[0,0] = 0.032168
差 = 2.25e-10  ← ε^2 オーダー


---
## 5. SGD パラメータ更新

**確率的勾配降下法（SGD）**：勾配の逆方向にパラメータを更新します：

$$W \leftarrow W - \eta \cdot \nabla_W \mathcal{L}$$

- $\eta$（学習率）：更新ステップの大きさ（小さすぎると遅い、大きすぎると発散）

In [8]:
learning_rate = 0.1

print("SGD 更新前後の比較（W_lm の最初の行）:")
print(f"  更新前 W_lm[0]: {W_lm[0]}")
print(f"  勾配 grad[0]:   {grad_W_lm[0]}")
print()

# 1 ステップ更新
W_lm_new = W_lm - learning_rate * grad_W_lm

print(f"  更新後 W_lm[0]: {W_lm_new[0]}")
print(f"  変化量:         {W_lm_new[0] - W_lm[0]}")
print()

# 更新後の損失を確認
logits_new = hidden @ W_lm_new
loss_new, _ = cross_entropy_loss(logits_new, target_ids)

print(f"更新前の損失: {loss:.6f}")
print(f"更新後の損失: {loss_new:.6f}")
print(f"損失の変化:  {loss_new - loss:.6f} ({'減少' if loss_new < loss else '増加'})")

SGD 更新前後の比較（W_lm の最初の行）:
  更新前 W_lm[0]: [-0.0427  0.6172  0.085   0.3989 -0.0464 -0.0207  0.2266  0.2477]
  勾配 grad[0]:   [ 0.0322  0.017   0.0565 -0.2257  0.0422  0.2114 -0.256   0.1223]

  更新後 W_lm[0]: [-0.0459  0.6155  0.0793  0.4215 -0.0506 -0.0419  0.2522  0.2355]
  変化量:         [-0.0032 -0.0017 -0.0056  0.0226 -0.0042 -0.0211  0.0256 -0.0122]

更新前の損失: 2.166881
更新後の損失: 2.061875
損失の変化:  -0.105006 (減少)


---
## 6. 小さなデータで学習ループを走らせる

W_lm のみを更新する簡易学習ループで、損失が下がる様子を確認します。

In [9]:
# W_lm のみを学習するシンプルなループ
np.random.seed(99)
W_lm_train = np.random.randn(d_model, vocab_size) * 0.3
lr     = 0.05
n_step = 30

loss_history = []

for step in range(n_step):
    # フォワードパス
    logits_t  = hidden @ W_lm_train
    loss_t, probs_t = cross_entropy_loss(logits_t, target_ids)
    loss_history.append(loss_t)

    # 勾配計算
    one_hot_t = np.zeros_like(probs_t)
    one_hot_t[np.arange(len(target_ids)), target_ids] = 1.0
    grad_t = hidden.T @ ((probs_t - one_hot_t) / len(target_ids))

    # パラメータ更新
    W_lm_train -= lr * grad_t

    if step % 5 == 0 or step == n_step - 1:
        print(f"Step {step:3d}: loss = {loss_t:.4f}")

print()
print(f"初期損失: {loss_history[0]:.4f}")
print(f"最終損失: {loss_history[-1]:.4f}")
print(f"→ 損失が下がり、モデルが正解を予測しやすくなった")

Step   0: loss = 2.1669
Step   5: loss = 1.9175
Step  10: loss = 1.7041
Step  15: loss = 1.5229
Step  20: loss = 1.3697
Step  25: loss = 1.2404
Step  29: loss = 1.1514

初期損失: 2.1669
最終損失: 1.1514
→ 損失が下がり、モデルが正解を予測しやすくなった


---
## 7. 自己回帰的テキスト生成

LLM のテキスト生成は「1 トークンずつ繰り返す」単純なループです：

```
1. プロンプト（初期トークン列）を入力
2. フォワードパスで logits を計算
3. 最後の位置の logits から次トークンを選択
4. 選択したトークンを系列に追加
5. 系列が最大長に達するまで 2〜4 を繰り返す
```

In [10]:
# 語彙マップ（仮のラベル）
idx2word = {0: "<PAD>", 1: "A", 2: "B", 3: "C", 4: "D", 5: "E", 6: "F", 7: "G"}

def greedy_generate(prompt_ids, max_new_tokens=6, verbose=True):
    """Greedy デコーディング: 毎ステップ最高確率トークンを選ぶ"""
    ids = list(prompt_ids)

    if verbose:
        print(f"プロンプト: {[idx2word[i] for i in ids]}")
        print()

    for step in range(max_new_tokens):
        logits_g, _ = model_forward(ids)
        last_logits = logits_g[-1]             # 最後の位置の logits
        next_probs  = softmax(last_logits)
        next_id     = int(np.argmax(next_probs)) # 最高確率

        if verbose:
            top3_ids = np.argsort(next_probs)[::-1][:3]
            print(f"  Step {step+1}: [{', '.join(idx2word[i] for i in ids)}]")
            print(f"    Top-3: ", end="")
            for tid in top3_ids:
                print(f"{idx2word[tid]}({next_probs[tid]:.3f})", end="  ")
            print(f"\n    → 選択: {idx2word[next_id]}")
            print()

        ids.append(next_id)
        if next_id == 0:   # <PAD> が出たら停止
            break

    return ids

generated = greedy_generate([2, 5], max_new_tokens=4)

プロンプト: ['B', 'E']

  Step 1: [B, E]
    Top-3: A(0.184)  C(0.163)  F(0.144)  
    → 選択: A

  Step 2: [B, E, A]
    Top-3: A(0.404)  G(0.181)  C(0.078)  
    → 選択: A

  Step 3: [B, E, A, A]
    Top-3: A(0.326)  G(0.200)  E(0.089)  
    → 選択: A

  Step 4: [B, E, A, A, A]
    Top-3: G(0.178)  E(0.173)  A(0.142)  
    → 選択: G



---
## 8. サンプリング戦略

Greedy は決定論的（常に同じ出力）。確率的なサンプリングで多様な出力を生成できます。

**Temperature**：logits を temperature で割ることで確率分布を調整

$$P_j = \frac{\exp(\text{logits}_j / T)}{\sum_k \exp(\text{logits}_k / T)}$$

- T < 1：分布が sharp になる（よりGreedyに近い）
- T = 1：通常の softmax
- T > 1：分布が flat になる（よりランダム）

In [11]:
# Temperature の効果を数値で確認
sample_logits = np.array([2.0, 1.0, 0.5, -0.5, -1.0, -1.5, -2.0, -2.5])

print("同じ logits に Temperature を変えて softmax を適用:")
print(f"logits = {sample_logits}")
print()

for T in [0.1, 0.5, 1.0, 2.0, 5.0]:
    probs_T = softmax(sample_logits / T)
    print(f"  T={T:3.1f}: {probs_T.round(3)}")
    print(f"         最大確率={probs_T.max():.3f}  entropy={-np.sum(probs_T * np.log(probs_T+1e-9)):.3f}")

同じ logits に Temperature を変えて softmax を適用:
logits = [ 2.   1.   0.5 -0.5 -1.  -1.5 -2.  -2.5]

  T=0.1: [1. 0. 0. 0. 0. 0. 0. 0.]
         最大確率=1.000  entropy=0.001
  T=0.5: [0.836 0.113 0.042 0.006 0.002 0.001 0.    0.   ]
         最大確率=0.836  entropy=0.579
  T=1.0: [0.561 0.206 0.125 0.046 0.028 0.017 0.01  0.006]
         最大確率=0.561  entropy=1.300
  T=2.0: [0.333 0.202 0.157 0.095 0.074 0.058 0.045 0.035]
         最大確率=0.333  entropy=1.820
  T=5.0: [0.197 0.162 0.146 0.12  0.108 0.098 0.089 0.08 ]
         最大確率=0.197  entropy=2.036


In [12]:
def temperature_sample(logits_1d, temperature=1.0, rng=None):
    """Temperature サンプリング"""
    if rng is None:
        rng = np.random.default_rng()
    probs = softmax(logits_1d / temperature)
    return int(rng.choice(len(probs), p=probs))


def top_k_sample(logits_1d, k=3, temperature=1.0, rng=None):
    """Top-k サンプリング: 上位 k 個の候補からのみサンプリング"""
    if rng is None:
        rng = np.random.default_rng()
    top_k_ids = np.argsort(logits_1d)[::-1][:k]   # 上位 k 個のインデックス
    top_k_logits = logits_1d[top_k_ids]
    top_k_probs  = softmax(top_k_logits / temperature)
    chosen_rank  = int(rng.choice(k, p=top_k_probs))
    return int(top_k_ids[chosen_rank])


def sampling_generate(prompt_ids, strategy="greedy", temperature=1.0, top_k=3,
                      max_new_tokens=5, seed=0):
    rng = np.random.default_rng(seed)
    ids = list(prompt_ids)
    for _ in range(max_new_tokens):
        logits_s, _ = model_forward(ids)
        last = logits_s[-1]
        if strategy == "greedy":
            next_id = int(np.argmax(softmax(last)))
        elif strategy == "temperature":
            next_id = temperature_sample(last, temperature, rng)
        elif strategy == "top_k":
            next_id = top_k_sample(last, k=top_k, temperature=temperature, rng=rng)
        ids.append(next_id)
        if next_id == 0:
            break
    return [idx2word[i] for i in ids]


prompt = [2, 5]
print(f"プロンプト: {[idx2word[i] for i in prompt]}")
print()
print("Greedy:")
print(" ", sampling_generate(prompt, strategy="greedy"))
print()
print("Temperature=0.5（より決定論的）:")
for trial in range(3):
    print(" ", sampling_generate(prompt, strategy="temperature", temperature=0.5, seed=trial))
print()
print("Temperature=2.0（よりランダム）:")
for trial in range(3):
    print(" ", sampling_generate(prompt, strategy="temperature", temperature=2.0, seed=trial))
print()
print("Top-k=3, Temperature=1.0:")
for trial in range(3):
    print(" ", sampling_generate(prompt, strategy="top_k", top_k=3, temperature=1.0, seed=trial))

プロンプト: ['B', 'E']

Greedy:
  ['B', 'E', 'A', 'A', 'A', 'G', 'D']

Temperature=0.5（より決定論的）:
  ['B', 'E', 'C', 'A', 'A', '<PAD>']
  ['B', 'E', 'C', 'G', 'A', 'G', 'D']
  ['B', 'E', 'A', 'A', 'G', 'A', 'E']

Temperature=2.0（よりランダム）:
  ['B', 'E', 'D', 'A', '<PAD>']
  ['B', 'E', 'C', 'G', 'A', 'G', 'B']
  ['B', 'E', 'A', 'A', 'F', '<PAD>']

Top-k=3, Temperature=1.0:
  ['B', 'E', 'C', 'A', 'A', 'E', '<PAD>']
  ['B', 'E', 'C', 'F', 'A', '<PAD>']
  ['B', 'E', 'A', 'A', 'G', 'D', 'E']


---
## 9. 生成の可視化（各ステップの確率分布）

In [13]:
# 各生成ステップの確率分布を表示
prompt_ids_vis = [2, 5]
n_gen = 4

print("各ステップの次トークン確率分布:")
print("=" * 60)

ids_vis = list(prompt_ids_vis)
for step in range(n_gen):
    logits_v, _ = model_forward(ids_vis)
    p = softmax(logits_v[-1])
    sorted_ids = np.argsort(p)[::-1]

    print(f"\nステップ {step+1}: 現在={[idx2word[i] for i in ids_vis]}")
    print("  次トークン確率:")
    for j in sorted_ids:
        bar = "█" * int(p[j] * 30)
        print(f"    {idx2word[j]:5s} ({j}): {p[j]:.4f} {bar}")

    next_id = int(np.argmax(p))
    print(f"  → Greedy 選択: {idx2word[next_id]}")
    ids_vis.append(next_id)

print()
print(f"最終生成: {[idx2word[i] for i in ids_vis]}")

各ステップの次トークン確率分布:

ステップ 1: 現在=['B', 'E']
  次トークン確率:
    A     (1): 0.1836 █████
    C     (3): 0.1630 ████
    F     (6): 0.1438 ████
    B     (2): 0.1430 ████
    <PAD> (0): 0.1205 ███
    G     (7): 0.1072 ███
    D     (4): 0.0770 ██
    E     (5): 0.0618 █
  → Greedy 選択: A

ステップ 2: 現在=['B', 'E', 'A']
  次トークン確率:
    A     (1): 0.4040 ████████████
    G     (7): 0.1808 █████
    C     (3): 0.0784 ██
    B     (2): 0.0737 ██
    F     (6): 0.0720 ██
    <PAD> (0): 0.0668 ██
    D     (4): 0.0642 █
    E     (5): 0.0600 █
  → Greedy 選択: A

ステップ 3: 現在=['B', 'E', 'A', 'A']
  次トークン確率:
    A     (1): 0.3263 █████████
    G     (7): 0.2004 ██████
    E     (5): 0.0885 ██
    <PAD> (0): 0.0857 ██
    D     (4): 0.0807 ██
    C     (3): 0.0767 ██
    B     (2): 0.0747 ██
    F     (6): 0.0670 ██
  → Greedy 選択: A

ステップ 4: 現在=['B', 'E', 'A', 'A', 'A']
  次トークン確率:
    G     (7): 0.1779 █████
    E     (5): 0.1734 █████
    A     (1): 0.1417 ████
    <PAD> (0): 0.1413 ████
    D     (4): 0.1256 ██

---
## まとめ：4 つのノートブック全体の流れ

```
テキスト
  ↓ [NB01] トークン化
整数列 → Embedding テーブル → X_embed
         Positional Encoding → PE
         X = X_embed + PE
  ↓
  X (seq_len, d_model)
  ↓ [NB02] Self-Attention
  Q, K, V = X @ W_{Q,K,V}
  scores = Q @ K.T / √d_k
  weights = softmax(scores + mask)
  attn_out = weights @ V
  ↓
  ↓ [NB03] Transformer ブロック × N
  X' = LayerNorm(X + MHA(X))
  X_out = LayerNorm(X' + FFN(X'))
  ↓
  logits = X_out @ W_lm
  ↓ [NB04] 学習 / 推論
  loss = -log P(y | x)
  ∇W = (P - one_hot) / N
  W ← W - η∇W
  ↓ 推論
  next_token = argmax(softmax(logits[-1]))
```

## 発展的なトピック

| トピック | 概要 |
|---------|------|
| **Adam** | モメンタム + 適応的学習率 → 実際に使われる最適化手法 |
| **RoPE** | 相対位置エンコーディング（LLaMA, GPT-NeoX 等） |
| **Flash Attention** | Attention のメモリ効率的な実装 |
| **KV Cache** | 推論の高速化（過去の K, V を再計算しない） |
| **Scaling Laws** | モデルサイズ・データ・計算量の関係 |
| **RLHF** | 人間のフィードバックによる強化学習 |

## 参考文献

- Vaswani et al. (2017) "Attention Is All You Need"
- Radford et al. (2018) "Improving Language Understanding by Generative Pre-Training" (GPT)
- Andrej Karpathy's nanoGPT: https://github.com/karpathy/nanoGPT